# 08 — Panchang-Derived Categorical Feature Engineering

## Purpose

Notebook 07 established the deterministic astronomical foundation for SOLETE:

- the canonical timeline is UTC;
- a 5-minute row at timestamp $r$ is represented astronomically at
  $r+2$ minutes;
- Skyfield with JPL DE440s supplies apparent geocentric Sun–Moon geometry;
- F2 contains local solar geometry;
- F3 contains continuous lunar and Sun–Moon astronomy.

This notebook constructs the **incremental categorical block used by F4**.

The study block contains:

1. Tithi;
2. Paksha;
3. Karana;
4. Nakshatra;
5. Yoga.

## Terminology: traditional Panchanga versus the study block

The traditional five limbs of Panchanga are **Tithi, Vaara, Nakshatra, Yoga,
and Karana**. Paksha is not one of the traditional five limbs.

Our incremental F4 block deliberately omits Vaara because ordinary weekday is
already represented in F1 and would otherwise duplicate that information.
Paksha is retained as an explicit two-state coarsening of the lunar phase.

Therefore the model block is called a **Panchang-derived categorical
representation**, not a literal encoding of all five traditional limbs.

## Instantaneous state rather than sunrise-assigned daily Panchanga

Traditional daily Panchanga publications usually identify the state prevailing
at local sunrise and report when that state ends. That convention is
appropriate for a daily almanac.

This project is a 5-minute forecasting study. We therefore use the
**instantaneous astronomical state** at the forecast-valid row's astronomical
reference instant. For a forecast issued at $t$ with horizon $h$,

$$
v=t+h
$$

is the target row and SOLETE F4 represents the state at

$$
v_{\mathrm{astro}}=t+h+2\text{ min}.
$$

The instantaneous Tithi, Paksha, Karana, Nakshatra, and Yoga states used here
are geocentric and do not require local sunrise. This is a forecasting
representation choice and should not be confused with assigning a traditional
civil/religious Panchanga day.

## Methodological references

The implementation is anchored to four types of reference:

1. **Government of India / Positional Astronomy Centre (PAC), Kolkata.**
   The *Rashtriya Panchang* is the standardized Indian Panchang and uses
   modern astronomical computation and Nirayana longitudes.
2. **Swiss Ephemeris documentation.**
   `SIDM_LAHIRI` is the standard Lahiri ayanamsha used by the Indian
   Astronomical Ephemeris since 1985. Swiss Ephemeris distinguishes this from
   the historical original ICRC value available separately as
   `SIDM_LAHIRI_ICRC`.
3. **Bokde et al. (2024), _VedicDateTime: An R package to implement Vedic
   calendar system_, DOI `10.1007/s11042-023-16553-w`.**
   This peer-reviewed implementation documents the standard Tithi,
   Nakshatra, Yoga, Karana, and Lahiri constructions and the sunrise-based
   daily convention.
4. **Drik Panchang** is used only as a secondary practical cross-reference for
   names and daily sunrise/end-time interpretation, not as the primary
   astronomical authority.

The principal equations are therefore treated as astronomical definitions,
while sunrise-based daily naming and festival rules remain outside F4.


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

import skyfield
import swisseph as swe
from skyfield.api import Loader
from skyfield.framelib import ecliptic_frame

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_solete

DATA_5MIN = PROJECT_ROOT / "solete_dataset" / "SOLETE_Pombo_5min.h5"
EPHEMERIS_DIR = PROJECT_ROOT / "ephemeris"
EPHEMERIS_FILE = EPHEMERIS_DIR / "de440s.bsp"

for path in [DATA_5MIN, EPHEMERIS_FILE]:
    if not path.exists():
        raise FileNotFoundError(path)

df_raw = load_solete(DATA_5MIN).sort_index()

assert df_raw.index.is_monotonic_increasing
assert not df_raw.index.duplicated().any()

ASTRONOMICAL_OFFSET_MINUTES = 2
astro_reference_times = (
    df_raw.index + pd.Timedelta(minutes=ASTRONOMICAL_OFFSET_MINUTES)
)

loader = Loader(str(EPHEMERIS_DIR))
ts = loader.timescale()
eph = loader("de440s.bsp")

earth = eph["Earth"]
moon = eph["Moon"]
sun = eph["Sun"]

skyfield_times = ts.from_datetimes(
    astro_reference_times.to_pydatetime()
)

print("Dataset:", df_raw.shape)
print("Period:", df_raw.index.min(), "to", df_raw.index.max())
print("Skyfield:", skyfield.__version__)
print("Swiss Ephemeris:", swe.version)
print("Astronomical row convention: timestamp + 2 minutes")


Dataset: (131617, 11)
Period: 2018-06-01 00:00:00+00:00 to 2019-09-01 00:00:00+00:00
Skyfield: 1.55
Swiss Ephemeris: 2.10.03
Astronomical row convention: timestamp + 2 minutes


## 1. Reuse the continuous astronomical state from F3

The Panchang-derived categories should be transformations of the same
underlying astronomy used by F3, rather than measurements from an additional
data source.

Skyfield therefore supplies apparent geocentric tropical/equinox-of-date
ecliptic longitudes of the Moon and Sun.

The Moon–Sun elongation is

$$
\phi=
(\lambda_{\mathrm{Moon}}-\lambda_{\mathrm{Sun}})
\bmod 360^\circ.
$$

Tithi, Paksha, and Karana depend only on $\phi$. A common sidereal offset
cancels from the longitude difference, so these three variables do not depend
on the selected ayanamsha.


In [2]:
earth_at_t = earth.at(skyfield_times)

moon_apparent = earth_at_t.observe(moon).apparent()
sun_apparent = earth_at_t.observe(sun).apparent()

moon_lat, moon_lon, moon_distance = (
    moon_apparent.frame_latlon(ecliptic_frame)
)
_, sun_lon, _ = sun_apparent.frame_latlon(ecliptic_frame)

moon_tropical_deg = moon_lon.degrees % 360.0
sun_tropical_deg = sun_lon.degrees % 360.0

phase_deg = (moon_tropical_deg - sun_tropical_deg) % 360.0

print("Rows:", len(phase_deg))
print("Moon tropical longitude range:",
      float(moon_tropical_deg.min()), "to", float(moon_tropical_deg.max()))
print("Sun tropical longitude range:",
      float(sun_tropical_deg.min()), "to", float(sun_tropical_deg.max()))
print("Moon–Sun phase range:",
      float(phase_deg.min()), "to", float(phase_deg.max()))


Rows: 131617
Moon tropical longitude range: 0.0027731837739083287 to 359.9985124103496
Sun tropical longitude range: 0.002462167065346934 to 359.999013619609
Moon–Sun phase range: 0.0004931953032212277 to 359.9996462129135


## 2. Standard Lahiri sidereal convention

Nakshatra and Yoga require a sidereal reference.

The canonical convention is Swiss Ephemeris **standard Lahiri**,
`SIDM_LAHIRI`. Swiss Ephemeris documents this as the official Lahiri
ayanamsha used by the Indian Astronomical Ephemeris from 1985 onward.

For the Panchang-derived implementation we follow the peer-reviewed
VedicDateTime construction more closely:

- obtain apparent tropical Sun/Moon longitude;
- obtain Lahiri ayanamsha with `get_ayanamsa_ex_ut()`;
- request `FLG_NONUT`, matching the package's published implementation;
- subtract that ayanamsha from the tropical longitude.

Thus

$$
\lambda_{\mathrm{sid}}
=
(\lambda_{\mathrm{trop}}-A_{\mathrm{Lahiri,NONUT}})
\bmod 360^\circ.
$$

This choice is now explicit because Swiss Ephemeris can also return an
ayanamsha including nutation. The two conventions differ only slightly, but
a small angular difference can change Nakshatra or Yoga at timestamps close
to a sector boundary.

Skyfield with JPL DE440s remains the canonical tropical-geometry source so
that F3 and F4 share the same underlying Sun–Moon state. Direct Swiss
Ephemeris calculations are retained below as an independent numerical
sensitivity check, not as a second definition of the canonical feature.

The historical `SIDM_LAHIRI_ICRC` convention is tested separately and is not
used for the 2018–2019 SOLETE features.


In [3]:
swe.set_sid_mode(swe.SIDM_LAHIRI)

def to_jd_ut(timestamp):
    hour = (
        timestamp.hour
        + timestamp.minute / 60.0
        + timestamp.second / 3600.0
        + timestamp.microsecond / 3_600_000_000.0
    )
    return swe.julday(
        timestamp.year,
        timestamp.month,
        timestamp.day,
        hour,
        swe.GREG_CAL,
    )

jd_full = np.array(
    [to_jd_ut(t) for t in astro_reference_times],
    dtype=float,
)

# Canonical VedicDateTime-style Lahiri value: NONUT is explicit.
lahiri_nonut_deg = np.empty(len(jd_full), dtype=float)
lahiri_retflags = np.empty(len(jd_full), dtype=int)

for i, jd_ut in enumerate(jd_full):
    retflag, value = swe.get_ayanamsa_ex_ut(
        jd_ut,
        swe.FLG_SWIEPH | swe.FLG_NONUT,
    )
    lahiri_retflags[i] = retflag
    lahiri_nonut_deg[i] = value

moon_sidereal_deg = (
    moon_tropical_deg - lahiri_nonut_deg
) % 360.0

sun_sidereal_deg = (
    sun_tropical_deg - lahiri_nonut_deg
) % 360.0

print(
    "Canonical Lahiri NONUT range:",
    float(lahiri_nonut_deg.min()),
    "to",
    float(lahiri_nonut_deg.max()),
)
print(
    "Unique ayanamsha return flags:",
    sorted(np.unique(lahiri_retflags).tolist()),
)
print(
    "Missing ayanamsha values:",
    int(np.isnan(lahiri_nonut_deg).sum()),
)


Canonical Lahiri NONUT range: 24.11431760121269 to 24.131796895735192
Unique ayanamsha return flags: [66]
Missing ayanamsha values: 0


## 3. Tithi, Paksha, and Karana

Tithi divides the Moon–Sun elongation into 30 sectors:

$$
T=
\left\lfloor \frac{\phi}{12^\circ}\right\rfloor+1,
\qquad
T\in\{1,\ldots,30\}.
$$

Paksha is the coarser phase state:

$$
\operatorname{Paksha}(T)=
\begin{cases}
\text{Shukla}, & 1\le T\le15,\\
\text{Krishna}, & 16\le T\le30.
\end{cases}
$$

Karana divides the same synodic cycle into 60 half-Tithi sectors:

$$
K=
\left\lfloor \frac{\phi}{6^\circ}\right\rfloor+1,
\qquad
K\in\{1,\ldots,60\}.
$$

The traditional Karana naming sequence is:

- slot 1: Kimstughna;
- slots 2–57: Bava, Balava, Kaulava, Taitila, Gara, Vanija, Vishti,
  repeated eight times;
- slots 58–60: Shakuni, Chatushpada, Naga.


In [4]:
TITHI_NAMES = {
    1: "Pratipada", 2: "Dvitiya", 3: "Tritiya", 4: "Chaturthi",
    5: "Panchami", 6: "Shashthi", 7: "Saptami", 8: "Ashtami",
    9: "Navami", 10: "Dashami", 11: "Ekadashi", 12: "Dvadashi",
    13: "Trayodashi", 14: "Chaturdashi",
}

MOVABLE_KARANAS = [
    "Bava", "Balava", "Kaulava", "Taitila",
    "Gara", "Vanija", "Vishti",
]

def tithi_name(tithi_in_paksha, paksha):
    if tithi_in_paksha == 15:
        return "Purnima" if paksha == "Shukla" else "Amavasya"
    return TITHI_NAMES[int(tithi_in_paksha)]

def karana_name(slot):
    slot = int(slot)
    if slot == 1:
        return "Kimstughna"
    if 2 <= slot <= 57:
        return MOVABLE_KARANAS[(slot - 2) % 7]
    return {58: "Shakuni", 59: "Chatushpada", 60: "Naga"}[slot]

tithi_number = np.floor(phase_deg / 12.0).astype(int) + 1
paksha = np.where(tithi_number <= 15, "Shukla", "Krishna")
tithi_in_paksha = (tithi_number - 1) % 15 + 1

karana_slot = np.floor(phase_deg / 6.0).astype(int) + 1
karana = np.array([karana_name(x) for x in karana_slot], dtype=object)

tithi_labels = np.array(
    [tithi_name(t, p) for t, p in zip(tithi_in_paksha, paksha)],
    dtype=object,
)

assert set(np.unique(tithi_number)) == set(range(1, 31))
assert set(np.unique(karana_slot)) == set(range(1, 61))
assert len(np.unique(karana)) == 11

boundary_phase = np.array([
    0.0, 12.0 - 1e-9, 12.0,
    180.0 - 1e-9, 180.0,
    360.0 - 1e-9, 360.0,
]) % 360.0

boundary_check = pd.DataFrame({
    "phase_deg": boundary_phase,
    "tithi": np.floor(boundary_phase / 12.0).astype(int) + 1,
    "karana_slot": np.floor(boundary_phase / 6.0).astype(int) + 1,
})

print("Tithi range:", tithi_number.min(), "to", tithi_number.max())
print("Paksha counts:")
print(pd.Series(paksha).value_counts())
print("Karana names:", sorted(np.unique(karana).tolist()))
display(boundary_check)


Tithi range: 1 to 30
Paksha counts:
Krishna    67545
Shukla     64072
Name: count, dtype: int64
Karana names: ['Balava', 'Bava', 'Chatushpada', 'Gara', 'Kaulava', 'Kimstughna', 'Naga', 'Shakuni', 'Taitila', 'Vanija', 'Vishti']


,phase_deg,tithi,karana_slot
0,0.0,1,1
1,12.0,1,2
2,12.0,2,3
3,180.0,15,30
4,180.0,16,31
5,360.0,30,60
6,0.0,1,1


## 4. Nakshatra and Yoga

Nakshatra divides the Lahiri sidereal lunar longitude into 27 equal sectors:

$$
N=
\left\lfloor
\frac{\lambda_{\mathrm{Moon,sid}}}{360^\circ/27}
\right\rfloor+1.
$$

Nitya Yoga uses the wrapped sum of the sidereal solar and lunar longitudes:

$$
\psi=
(\lambda_{\mathrm{Sun,sid}}+\lambda_{\mathrm{Moon,sid}})
\bmod360^\circ,
$$

followed by

$$
Y=
\left\lfloor\frac{\psi}{360^\circ/27}\right\rfloor+1.
$$

Both sectors have width

$$
\frac{360^\circ}{27}=13^\circ20'.
$$


In [5]:
NAKSHATRA_NAMES = [
    "Ashwini", "Bharani", "Krittika", "Rohini", "Mrigashirsha",
    "Ardra", "Punarvasu", "Pushya", "Ashlesha", "Magha",
    "Purva Phalguni", "Uttara Phalguni", "Hasta", "Chitra", "Swati",
    "Vishakha", "Anuradha", "Jyeshtha", "Mula", "Purva Ashadha",
    "Uttara Ashadha", "Shravana", "Dhanishta", "Shatabhisha",
    "Purva Bhadrapada", "Uttara Bhadrapada", "Revati",
]

YOGA_NAMES = [
    "Vishkambha", "Priti", "Ayushman", "Saubhagya", "Shobhana",
    "Atiganda", "Sukarma", "Dhriti", "Shula", "Ganda", "Vriddhi",
    "Dhruva", "Vyaghata", "Harshana", "Vajra", "Siddhi", "Vyatipata",
    "Variyana", "Parigha", "Shiva", "Siddha", "Sadhya", "Shubha",
    "Shukla", "Brahma", "Indra", "Vaidhriti",
]

SECTOR_27_DEG = 360.0 / 27.0

nakshatra_number = (
    np.floor(moon_sidereal_deg / SECTOR_27_DEG).astype(int) + 1
)
nakshatra = np.array(
    [NAKSHATRA_NAMES[n - 1] for n in nakshatra_number],
    dtype=object,
)

yoga_angle_deg = (
    moon_sidereal_deg + sun_sidereal_deg
) % 360.0
yoga_number = (
    np.floor(yoga_angle_deg / SECTOR_27_DEG).astype(int) + 1
)
yoga = np.array(
    [YOGA_NAMES[n - 1] for n in yoga_number],
    dtype=object,
)

assert set(np.unique(nakshatra_number)) == set(range(1, 28))
assert set(np.unique(yoga_number)) == set(range(1, 28))

boundary_27 = np.array([
    0.0,
    SECTOR_27_DEG - 1e-9,
    SECTOR_27_DEG,
    360.0 - 1e-9,
    360.0,
]) % 360.0

boundary_27_check = pd.DataFrame({
    "angle_deg": boundary_27,
    "sector_number":
        np.floor(boundary_27 / SECTOR_27_DEG).astype(int) + 1,
})

print("Nakshatra range:",
      nakshatra_number.min(), "to", nakshatra_number.max())
print("Yoga range:", yoga_number.min(), "to", yoga_number.max())
print("Unique Nakshatras:", len(np.unique(nakshatra)))
print("Unique Yogas:", len(np.unique(yoga)))
display(boundary_27_check)


Nakshatra range: 1 to 27
Yoga range: 1 to 27
Unique Nakshatras: 27
Unique Yogas: 27


,angle_deg,sector_number
0,0.000000,1
1,13.333333,1
2,13.333333,2
3,360.000000,27
4,0.000000,1


## 5. Independent full-timeline ephemeris validation

The canonical F4 construction intentionally follows the published
VedicDateTime-style coordinate convention:

1. compute apparent tropical Sun/Moon longitude;
2. compute standard Lahiri ayanamsha with `FLG_NONUT`;
3. subtract that ayanamsha from the tropical longitude.

To validate the astronomical engine without simultaneously changing the
calendrical convention, the independent reference must use the **same
transformation**.

For every SOLETE row we therefore compare:

- Skyfield/JPL DE440s apparent tropical Sun/Moon longitudes;
- Swiss Ephemeris API tropical Sun/Moon longitudes;
- after applying the same canonical Lahiri-NONUT ayanamsha to both,
  the resulting Nakshatra and Yoga parent angles;
- the final Tithi, Karana, Nakshatra, and Yoga category numbers.

This isolates differences due to the underlying solar/lunar ephemeris rather
than mixing them with a different sidereal/nutation convention.

The Swiss API may fall back from requested Swiss ephemeris files to its
built-in Moshier calculation. The returned flags are therefore reported
explicitly.

A few category differences are possible only when an actual timestamp lies
extremely close to a sector boundary. If any occur, they are printed with
their timestamps for inspection rather than being hidden.


In [6]:
def circular_difference_deg(a, b):
    return np.abs(
        (np.asarray(a) - np.asarray(b) + 180.0) % 360.0 - 180.0
    )

SWE_TROPICAL_FLAGS = swe.FLG_SWIEPH
n_rows = len(df_raw)

swiss_sun_tropical = np.empty(n_rows, dtype=float)
swiss_moon_tropical = np.empty(n_rows, dtype=float)
swiss_sun_flags = np.empty(n_rows, dtype=int)
swiss_moon_flags = np.empty(n_rows, dtype=int)

for i, jd_ut in enumerate(jd_full):
    s_t, s_flag = swe.calc_ut(
        jd_ut, swe.SUN, SWE_TROPICAL_FLAGS
    )
    m_t, m_flag = swe.calc_ut(
        jd_ut, swe.MOON, SWE_TROPICAL_FLAGS
    )

    swiss_sun_tropical[i] = s_t[0]
    swiss_moon_tropical[i] = m_t[0]
    swiss_sun_flags[i] = s_flag
    swiss_moon_flags[i] = m_flag

# Apply exactly the same canonical Lahiri-NONUT transformation used by F4.
swiss_sun_sidereal_same_convention = (
    swiss_sun_tropical - lahiri_nonut_deg
) % 360.0
swiss_moon_sidereal_same_convention = (
    swiss_moon_tropical - lahiri_nonut_deg
) % 360.0

cross_engine_angle_summary = pd.Series(
    {
        "max_sun_tropical_error_deg":
            float(
                circular_difference_deg(
                    sun_tropical_deg,
                    swiss_sun_tropical,
                ).max()
            ),
        "max_moon_tropical_error_deg":
            float(
                circular_difference_deg(
                    moon_tropical_deg,
                    swiss_moon_tropical,
                ).max()
            ),
        "max_sun_sidereal_same_convention_error_deg":
            float(
                circular_difference_deg(
                    sun_sidereal_deg,
                    swiss_sun_sidereal_same_convention,
                ).max()
            ),
        "max_moon_sidereal_same_convention_error_deg":
            float(
                circular_difference_deg(
                    moon_sidereal_deg,
                    swiss_moon_sidereal_same_convention,
                ).max()
            ),
    },
    name="full-timeline angular validation",
)

# Apply the same categorical definitions to the independent ephemeris route.
swiss_phase_deg = (
    swiss_moon_tropical - swiss_sun_tropical
) % 360.0

swiss_tithi_number = (
    np.floor(swiss_phase_deg / 12.0).astype(int) + 1
)
swiss_karana_slot = (
    np.floor(swiss_phase_deg / 6.0).astype(int) + 1
)

swiss_nakshatra_number = (
    np.floor(
        swiss_moon_sidereal_same_convention / SECTOR_27_DEG
    ).astype(int) + 1
)

swiss_yoga_angle_deg = (
    swiss_moon_sidereal_same_convention
    + swiss_sun_sidereal_same_convention
) % 360.0
swiss_yoga_number = (
    np.floor(
        swiss_yoga_angle_deg / SECTOR_27_DEG
    ).astype(int) + 1
)

mismatch_mask = (
    (tithi_number != swiss_tithi_number)
    | (karana_slot != swiss_karana_slot)
    | (nakshatra_number != swiss_nakshatra_number)
    | (yoga_number != swiss_yoga_number)
)

cross_engine_category_summary = pd.Series(
    {
        "rows_compared": n_rows,
        "Tithi_mismatches":
            int((tithi_number != swiss_tithi_number).sum()),
        "Karana_slot_mismatches":
            int((karana_slot != swiss_karana_slot).sum()),
        "Nakshatra_mismatches":
            int(
                (
                    nakshatra_number
                    != swiss_nakshatra_number
                ).sum()
            ),
        "Yoga_mismatches":
            int((yoga_number != swiss_yoga_number).sum()),
        "rows_with_any_category_mismatch":
            int(mismatch_mask.sum()),
    },
    name="full-timeline category validation",
)

sample_positions = np.linspace(0, n_rows - 1, 6, dtype=int)

sample_validation = pd.DataFrame(
    {
        "timestamp": astro_reference_times[sample_positions],
        "sun_tropical_error_deg":
            circular_difference_deg(
                sun_tropical_deg[sample_positions],
                swiss_sun_tropical[sample_positions],
            ),
        "moon_tropical_error_deg":
            circular_difference_deg(
                moon_tropical_deg[sample_positions],
                swiss_moon_tropical[sample_positions],
            ),
        "moon_sidereal_same_convention_error_deg":
            circular_difference_deg(
                moon_sidereal_deg[sample_positions],
                swiss_moon_sidereal_same_convention[sample_positions],
            ),
    }
)

print("Representative rows:")
display(sample_validation)

print("Complete-timeline angular comparison:")
display(cross_engine_angle_summary.to_frame("value"))

print("Complete-timeline categorical comparison:")
display(cross_engine_category_summary.to_frame("value"))

flag_summary = pd.DataFrame(
    {
        "sun": pd.Series(swiss_sun_flags).value_counts(),
        "moon": pd.Series(swiss_moon_flags).value_counts(),
    }
).fillna(0).astype(int)

print("Swiss API return-flag counts:")
display(flag_summary)

if mismatch_mask.any():
    mismatch_idx = np.flatnonzero(mismatch_mask)

    mismatch_detail = pd.DataFrame(
        {
            "row_timestamp": df_raw.index[mismatch_idx],
            "astronomical_timestamp":
                astro_reference_times[mismatch_idx],
            "canonical_tithi": tithi_number[mismatch_idx],
            "reference_tithi": swiss_tithi_number[mismatch_idx],
            "canonical_karana_slot": karana_slot[mismatch_idx],
            "reference_karana_slot": swiss_karana_slot[mismatch_idx],
            "canonical_nakshatra":
                nakshatra_number[mismatch_idx],
            "reference_nakshatra":
                swiss_nakshatra_number[mismatch_idx],
            "canonical_yoga": yoga_number[mismatch_idx],
            "reference_yoga": swiss_yoga_number[mismatch_idx],
        }
    )

    print("Rows with at least one category mismatch:")
    display(mismatch_detail)
else:
    print("No categorical mismatches across the complete timeline.")


Representative rows:


,timestamp,sun_tropical_error_deg,moon_tropical_error_deg,moon_sidereal_same_convention_error_deg
0,2018-06-01 00:02:00+00:00,0.000016,0.000009,0.000009
1,2018-08-31 09:37:00+00:00,0.000004,0.000292,0.000292
2,2018-11-30 19:12:00+00:00,0.000015,0.000016,0.000016
3,2019-03-02 04:47:00+00:00,0.000001,0.000075,0.000075
4,2019-06-01 14:22:00+00:00,0.000010,0.000092,0.000092
5,2019-09-01 00:02:00+00:00,0.000003,0.000084,0.000084


Complete-timeline angular comparison:


,value
max_sun_tropical_error_deg,0.000021
max_moon_tropical_error_deg,0.000474
max_sun_sidereal_same_convention_error_deg,0.000021
max_moon_sidereal_same_convention_error_deg,0.000474


Complete-timeline categorical comparison:


,value
rows_compared,131617
Tithi_mismatches,3
Karana_slot_mismatches,4
Nakshatra_mismatches,2
Yoga_mismatches,1
rows_with_any_category_mismatch,7


Swiss API return-flag counts:


,sun,moon
4,131617,131617


Rows with at least one category mismatch:


,row_timestamp,astronomical_timestamp,canonical_tithi,reference_tithi,canonical_karana_slot,reference_karana_slot,canonical_nakshatra,reference_nakshatra,canonical_yoga,reference_yoga
0,2018-07-05 19:35:00+00:00,2018-07-05 19:37:00+00:00,23,22,45,44,26,26,5,5
1,2018-09-21 21:50:00+00:00,2018-09-21 21:52:00+00:00,12,12,24,24,23,23,7,8
2,2018-10-11 00:35:00+00:00,2018-10-11 00:37:00+00:00,3,2,5,4,15,15,1,1
3,2019-02-14 20:40:00+00:00,2019-02-14 20:42:00+00:00,10,10,20,19,5,5,27,27
4,2019-02-18 08:30:00+00:00,2019-02-18 08:32:00+00:00,14,14,27,27,8,9,4,4
5,2019-02-21 08:30:00+00:00,2019-02-21 08:32:00+00:00,17,18,34,35,12,12,8,8
6,2019-05-06 11:05:00+00:00,2019-05-06 11:07:00+00:00,2,2,4,4,3,4,5,5


## 6. Sensitivity to the nutation convention

The canonical implementation above follows the published VedicDateTime code
and requests Lahiri ayanamsha with `FLG_NONUT`.

Swiss Ephemeris can also return the ayanamsha including nutation. This
section quantifies how often that alternative would change the two
ayanamsha-dependent categories:

- Nakshatra;
- Yoga.

Tithi, Paksha, and Karana do not need to be recomputed because a common
ayanamsha cancels from the Moon–Sun longitude difference.

This is a methodological sensitivity analysis only. The canonical F4 labels
remain the `FLG_NONUT` labels.


In [7]:
lahiri_with_nutation_deg = np.empty(len(jd_full), dtype=float)

for i, jd_ut in enumerate(jd_full):
    _, value = swe.get_ayanamsa_ex_ut(
        jd_ut,
        swe.FLG_SWIEPH,
    )
    lahiri_with_nutation_deg[i] = value

moon_sidereal_with_nutation = (
    moon_tropical_deg - lahiri_with_nutation_deg
) % 360.0
sun_sidereal_with_nutation = (
    sun_tropical_deg - lahiri_with_nutation_deg
) % 360.0

nakshatra_with_nutation = (
    np.floor(
        moon_sidereal_with_nutation / SECTOR_27_DEG
    ).astype(int) + 1
)

yoga_angle_with_nutation = (
    moon_sidereal_with_nutation
    + sun_sidereal_with_nutation
) % 360.0
yoga_with_nutation = (
    np.floor(
        yoga_angle_with_nutation / SECTOR_27_DEG
    ).astype(int) + 1
)

nak_change = nakshatra_with_nutation != nakshatra_number
yoga_change = yoga_with_nutation != yoga_number

nutation_sensitivity = pd.Series(
    {
        "max_ayanamsha_difference_deg":
            float(
                np.abs(
                    lahiri_with_nutation_deg - lahiri_nonut_deg
                ).max()
            ),
        "max_ayanamsha_difference_arcsec":
            float(
                np.abs(
                    lahiri_with_nutation_deg - lahiri_nonut_deg
                ).max() * 3600.0
            ),
        "Nakshatra_changed_rows":
            int(nak_change.sum()),
        "Yoga_changed_rows":
            int(yoga_change.sum()),
        "rows_changed_in_either":
            int((nak_change | yoga_change).sum()),
    },
    name="NONUT-vs-nutation Lahiri sensitivity",
)

display(nutation_sensitivity.to_frame("value"))


,value
max_ayanamsha_difference_deg,0.004906
max_ayanamsha_difference_arcsec,17.660599
Nakshatra_changed_rows,43.000000
Yoga_changed_rows,87.000000
rows_changed_in_either,130.000000


## 7. Historical ICRC Lahiri versus the modern standard Lahiri

Swiss Ephemeris distinguishes:

- `SIDM_LAHIRI`: the standard Lahiri used by the Indian Astronomical
  Ephemeris from 1985 onward;
- `SIDM_LAHIRI_ICRC`: the historical Calendar Reform Committee value.

Because SOLETE covers 2018–2019, the post-1985 standard Lahiri definition is
canonical for this project.

To isolate only the sidereal-zero-point choice, both variants are evaluated
with the same `FLG_NONUT` convention used by the canonical implementation.
This is a sensitivity analysis and does not alter F4.


In [8]:
if hasattr(swe, "SIDM_LAHIRI_ICRC"):
    swe.set_sid_mode(swe.SIDM_LAHIRI_ICRC)

    lahiri_icrc_nonut_deg = np.array(
        [
            swe.get_ayanamsa_ex_ut(
                jd,
                swe.FLG_SWIEPH | swe.FLG_NONUT,
            )[1]
            for jd in jd_full
        ],
        dtype=float,
    )

    moon_sidereal_icrc = (
        moon_tropical_deg - lahiri_icrc_nonut_deg
    ) % 360.0
    sun_sidereal_icrc = (
        sun_tropical_deg - lahiri_icrc_nonut_deg
    ) % 360.0

    nakshatra_icrc = (
        np.floor(
            moon_sidereal_icrc / SECTOR_27_DEG
        ).astype(int) + 1
    )

    yoga_angle_icrc = (
        moon_sidereal_icrc + sun_sidereal_icrc
    ) % 360.0
    yoga_icrc = (
        np.floor(
            yoga_angle_icrc / SECTOR_27_DEG
        ).astype(int) + 1
    )

    historical_lahiri_sensitivity = pd.Series(
        {
            "max_standard_minus_ICRC_arcsec":
                float(
                    np.max(
                        np.abs(
                            lahiri_nonut_deg
                            - lahiri_icrc_nonut_deg
                        )
                    ) * 3600.0
                ),
            "Nakshatra_changed_rows":
                int(
                    (nakshatra_icrc != nakshatra_number).sum()
                ),
            "Yoga_changed_rows":
                int(
                    (yoga_icrc != yoga_number).sum()
                ),
            "rows_changed_in_either":
                int(
                    (
                        (nakshatra_icrc != nakshatra_number)
                        | (yoga_icrc != yoga_number)
                    ).sum()
                ),
        },
        name="standard-vs-historical Lahiri sensitivity",
    )

    display(
        historical_lahiri_sensitivity.to_frame("value")
    )

    # Restore the canonical post-1985 standard.
    swe.set_sid_mode(swe.SIDM_LAHIRI)
else:
    print(
        "SIDM_LAHIRI_ICRC is not exposed by this Swiss Ephemeris build; "
        "historical-Lahiri sensitivity skipped."
    )


,value
max_standard_minus_ICRC_arcsec,1.092015
Nakshatra_changed_rows,4.000000
Yoga_changed_rows,4.000000
rows_changed_in_either,8.000000


## 8. Canonical raw F4 block and fixed category schema

The modelling block contains categorical states only. Continuous intermediates
remain in a separate audit table.

The category universe is known **a priori** from the definitions:

- 30 Tithis;
- 2 Pakshas;
- 11 Karana names;
- 27 Nakshatras;
- 27 Yogas.

Therefore a future one-hot representation does not need to *discover*
categories from the training data. A fixed full one-hot schema would contain

$$
30+2+11+27+27=97
$$

indicator columns.

That fixed schema does not leak target or future measured information because
the domains are mathematical definitions, not statistics estimated from the
dataset. Reference-category coding can later be tested as a sensitivity
analysis.

Paksha is perfectly determined by Tithi and is intentionally retained as an
explicit coarser representation. Its contribution can later be examined in
component ablations.


In [9]:
KARANA_CATEGORIES = [
    "Kimstughna",
    "Bava", "Balava", "Kaulava", "Taitila",
    "Gara", "Vanija", "Vishti",
    "Shakuni", "Chatushpada", "Naga",
]

category_schema = {
    "tithi": list(range(1, 31)),
    "paksha": ["Shukla", "Krishna"],
    "karana": KARANA_CATEGORIES,
    "nakshatra": NAKSHATRA_NAMES,
    "yoga": YOGA_NAMES,
}

df_panchang_f4_raw = pd.DataFrame(
    {
        "tithi": tithi_number,
        "paksha": paksha,
        "karana": karana,
        "nakshatra": nakshatra,
        "yoga": yoga,
    },
    index=df_raw.index,
)

for col, categories in category_schema.items():
    dtype = pd.CategoricalDtype(categories=categories, ordered=False)
    df_panchang_f4_raw[col] = df_panchang_f4_raw[col].astype(dtype)

df_panchang_audit = pd.DataFrame(
    {
        "lahiri_ayanamsha_nonut_deg": lahiri_nonut_deg,
        "sun_tropical_longitude_deg": sun_tropical_deg,
        "moon_tropical_longitude_deg": moon_tropical_deg,
        "sun_sidereal_longitude_deg": sun_sidereal_deg,
        "moon_sidereal_longitude_deg": moon_sidereal_deg,
        "moon_sun_phase_deg": phase_deg,
        "tithi_number": tithi_number,
        "tithi_name": tithi_labels,
        "paksha": paksha,
        "karana_slot": karana_slot,
        "karana_name": karana,
        "nakshatra_number": nakshatra_number,
        "nakshatra_name": nakshatra,
        "yoga_angle_deg": yoga_angle_deg,
        "yoga_number": yoga_number,
        "yoga_name": yoga,
    },
    index=df_raw.index,
)

EXPECTED_FULL_ONEHOT_WIDTH = sum(
    len(v) for v in category_schema.values()
)

verification = pd.Series({
    "rows": len(df_panchang_f4_raw),
    "raw_F4_variables": df_panchang_f4_raw.shape[1],
    "raw_F4_missing": int(df_panchang_f4_raw.isna().sum().sum()),
    "audit_missing": int(df_panchang_audit.isna().sum().sum()),
    "index_matches_SOLETE":
        bool(df_panchang_f4_raw.index.equals(df_raw.index)),
    "expected_full_onehot_width": EXPECTED_FULL_ONEHOT_WIDTH,
}, name="F4 verification")

display(verification.to_frame("value"))
display(df_panchang_f4_raw.head(12))

assert EXPECTED_FULL_ONEHOT_WIDTH == 97
assert verification["raw_F4_missing"] == 0
assert verification["audit_missing"] == 0
assert verification["index_matches_SOLETE"]


,value
rows,131617
raw_F4_variables,5
raw_F4_missing,0
audit_missing,0
index_matches_SOLETE,True
expected_full_onehot_width,97


,tithi,paksha,karana,nakshatra,yoga
Timestamp,,,,,
2018-06-01 00:00:00+00:00,18,Krishna,Vanija,Mula,Shubha
2018-06-01 00:05:00+00:00,18,Krishna,Vanija,Mula,Shubha
2018-06-01 00:10:00+00:00,18,Krishna,Vanija,Mula,Shubha
2018-06-01 00:15:00+00:00,18,Krishna,Vanija,Mula,Shubha
2018-06-01 00:20:00+00:00,18,Krishna,Vanija,Mula,Shubha
2018-06-01 00:25:00+00:00,18,Krishna,Vanija,Purva Ashadha,Shubha
2018-06-01 00:30:00+00:00,18,Krishna,Vanija,Purva Ashadha,Shubha
2018-06-01 00:35:00+00:00,18,Krishna,Vanija,Purva Ashadha,Shubha
2018-06-01 00:40:00+00:00,18,Krishna,Vanija,Purva Ashadha,Shubha


## 9. A stricter incremental continuous-parent control for later modelling

There is one subtle representation issue worth preserving explicitly.

The current F3 block from Notebook 07 already contains the continuous
Moon–Sun phase angle as sine/cosine, together with tropical Sun and Moon
longitudes. Therefore the continuous parent of Tithi, Paksha, and Karana is
**already controlled in F3** and should not be appended a second time.

Nakshatra and Yoga, however, introduce the Lahiri sidereal frame before
discretization. An observed F3 → F4 gain could therefore contain two effects:

1. introducing those sidereal continuous angles;
2. discretizing them into Panchang-derived categories.

To separate the effects later, this notebook creates an optional
**incremental continuous-parent control** containing only the two continuous
parents not already present explicitly in F3:

- Lahiri sidereal lunar longitude;
- Lahiri sidereal Yoga angle.

Each is represented by sine/cosine, giving four additional variables.

This table is **not part of F4** and does not alter the main hierarchy. It is
retained for a later robustness comparison:

$$
F3^\ast
=
F3
+
C(\lambda_{\mathrm{Moon,sid}})
+
C(\psi_{\mathrm{Yoga}}).
$$

A comparison between $F3^\ast$ and $F3^\ast+$ categorical F4 isolates the
discretization/representation question more tightly without duplicating the
phase variables already contained in F3.


In [10]:
df_panchang_parent_continuous = pd.DataFrame(
    index=df_raw.index
)

for angle, prefix in [
    (moon_sidereal_deg, "moon_sidereal"),
    (yoga_angle_deg, "yoga_angle"),
]:
    radians = np.deg2rad(angle)
    df_panchang_parent_continuous[f"{prefix}_sin"] = np.sin(radians)
    df_panchang_parent_continuous[f"{prefix}_cos"] = np.cos(radians)

parent_circle_errors = {}

for prefix in ["moon_sidereal", "yoga_angle"]:
    s = df_panchang_parent_continuous[f"{prefix}_sin"]
    c = df_panchang_parent_continuous[f"{prefix}_cos"]
    parent_circle_errors[prefix] = float(
        np.abs(s**2 + c**2 - 1).max()
    )

print(
    "Incremental continuous-parent control shape:",
    df_panchang_parent_continuous.shape,
)
print(
    "Missing values:",
    int(df_panchang_parent_continuous.isna().sum().sum()),
)

display(
    pd.Series(
        parent_circle_errors,
        name="max_unit_circle_error",
    ).to_frame()
)

assert df_panchang_parent_continuous.shape[1] == 4
assert not df_panchang_parent_continuous.isna().any().any()


Incremental continuous-parent control shape: (131617, 4)
Missing values: 0


,max_unit_circle_error
moon_sidereal,2.220446e-16
yoga_angle,2.220446e-16


## Final protocol

Notebook 08 now fixes the F4 feature-generation rules without fitting a model:

| Quantity | Definition | Ayanamsha dependence |
|---|---|---|
| Tithi | 30 sectors of Moon–Sun elongation | no |
| Paksha | waxing/waning state derived from Tithi | no |
| Karana | 60 half-Tithi sectors mapped to 11 names | no |
| Nakshatra | 27 sectors of Lahiri sidereal lunar longitude | yes |
| Yoga | 27 sectors of the Lahiri sidereal Sun+Moon sum | yes |

The canonical F4 block is the five-column `df_panchang_f4_raw`. Its category
schema is fixed in advance and corresponds to 97 columns under full one-hot
coding.

Two methodological controls are retained separately:

- the NONUT-vs-nutation Lahiri sensitivity;
- `df_panchang_parent_continuous`, a four-variable incremental control that
  adds the sidereal Moon and Yoga parent angles not already explicit in F3.

Masa remains outside the baseline because it requires additional month-boundary
and naming conventions.

Before freezing F4, the executed full-timeline independent-ephemeris
category table should be inspected. The comparison holds the
Lahiri-NONUT convention fixed, so any remaining category differences are
true boundary sensitivity to the solar/lunar ephemeris rather than a
mixture of ephemeris and sidereal-convention changes.

The next stage should move into reusable feature assembly and forecasting
sample construction. That is also where rolling/blocked temporal
cross-validation, fixed forecast-valid-time assignment, training-fold-only
scaling, and the final model suite should be frozen.
